# RAG Pipeline: Docling + Ray Data + Milvus + vLLM

End-to-end **Retrieval-Augmented Generation** on **Red Hat OpenShift AI**.

## What this notebook does

1. **Parse & Chunk PDFs** — Ray Data distributes Docling processing across workers. Docling's `HybridChunker` splits documents using structural boundaries (headings, paragraphs, tables).
2. **Embed & Ingest** — Each actor generates embeddings with `sentence-transformers` and inserts directly into Milvus. No intermediate files — the PVC is read-only.
3. **Deploy a Model** — An LLM is served via vLLM on OpenShift AI.
4. **RAG Query** — Search Milvus for relevant chunks, build a prompt with context, and query the LLM.

## Architecture

```
  Input PDFs (read-only PVC)
         │
         ▼
  ┌─────────────────────────────────────────────┐
  │  RayJob (N workers × M actors each)         │
  │                                             │
  │  Actor:  Docling parse                      │
  │          → HybridChunker                    │
  │          → sentence-transformers embed      │
  │          → Milvus insert                    │
  └─────────────────────────────────────────────┘
         │                          │
         ▼                          ▼
    Milvus DB                  vLLM Endpoint
    (vectors)                 (InferenceService)
         │                          │
         └──────────┬───────────────┘
                    ▼
             RAG Query Loop
          (search → prompt → answer)
```

## Prerequisites

- OpenShift cluster with **RHOAI** installed
- **KubeRay** operator enabled
- **Milvus** deployed (see `../milvus-ocp/`)
- A **ReadWriteMany PVC** with input PDFs (read-only access is sufficient)
- A container image with Ray + Docling + pymilvus + sentence-transformers
- GPU nodes for model serving (or a smaller CPU-compatible model)

---
## Step 1 — Install Dependencies

In [ ]:
!pip install -q codeflare-sdk kubernetes "ray[default]" pymilvus sentence-transformers openai

In [ ]:
import json
import os
import subprocess
import time

from codeflare_sdk import ManagedClusterConfig, RayJob
from kubernetes import client as kclient
from kubernetes.client import (
    V1PersistentVolumeClaimVolumeSource,
    V1Volume,
    V1VolumeMount,
)
from kubernetes.client.rest import ApiException
from ray.runtime_env import RuntimeEnv

---
## Step 2 — Connect to OpenShift

Replace `TOKEN` and `API_URL` with your own credentials. Get a token from the OpenShift web console under **Copy login command**.

In [ ]:
# ---- REPLACE THESE WITH YOUR VALUES ----
TOKEN = "sha256~your-token-here"
API_URL = "https://api.your-cluster.example.com:443"
NAMESPACE = "rag-example"

In [ ]:
!oc login --token={TOKEN} --server={API_URL}

---
## Step 3 — Configure Parameters

### Cluster sizing

| Parameter | Formula |
|---|---|
| Schedulable CPUs per worker | `worker_cpus - 2` (2 reserved for Ray overhead) |
| Actors per worker | `schedulable_cpus // cpus_per_actor` |
| Max actors (cluster-wide) | `num_workers × actors_per_worker` |

### Chunking

Docling's `HybridChunker` uses document structure (headings, paragraphs, tables) to create semantically meaningful chunks. `CHUNK_MAX_TOKENS` controls the maximum chunk length using the embedding model's tokenizer.

In [ ]:
# ---------------------------------------------------------------------------
# Cluster sizing
# ---------------------------------------------------------------------------
NUM_WORKERS = 2
WORKER_CPUS = 8
WORKER_MEMORY_GB = 16
HEAD_CPUS = 2
HEAD_MEMORY_GB = 4

# ---------------------------------------------------------------------------
# Actor configuration
# ---------------------------------------------------------------------------
CPUS_PER_ACTOR = 4
MIN_ACTORS = 2
MAX_ACTORS = 4
BATCH_SIZE = 4

# ---------------------------------------------------------------------------
# Chunking (Docling HybridChunker)
# ---------------------------------------------------------------------------
CHUNK_MAX_TOKENS = 256  # Max tokens per chunk (uses embedding model tokenizer)

# ---------------------------------------------------------------------------
# Embedding
# ---------------------------------------------------------------------------
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# ---------------------------------------------------------------------------
# Milvus
# ---------------------------------------------------------------------------
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
MILVUS_DB = "default"
COLLECTION_NAME = "rag_documents"

# ---------------------------------------------------------------------------
# Storage (read-only PVC)
# ---------------------------------------------------------------------------
PVC_NAME = "data-pvc"
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/pdfs"

# ---------------------------------------------------------------------------
# Container image with Ray + Docling + pymilvus + sentence-transformers
# ---------------------------------------------------------------------------
IMAGE = "quay.io/rhoai-szaher/docling-ray:latest"

# ---------------------------------------------------------------------------
# Timeout and fault tolerance
# ---------------------------------------------------------------------------
TIMEOUT_SECONDS = 600
NUM_FILES = 0  # 0 = all

# ---------------------------------------------------------------------------
# RayJob
# ---------------------------------------------------------------------------
RAYJOB_NAME = f"docling-milvus-{int(time.time())}"
TTL_AFTER_FINISHED = 300

# ---------------------------------------------------------------------------
# Model serving
# ---------------------------------------------------------------------------
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
GPU_COUNT = 1

In [ ]:
# Derived values
SCHEDULABLE_CPUS = WORKER_CPUS - 2
ACTORS_PER_WORKER = SCHEDULABLE_CPUS // CPUS_PER_ACTOR

print(f"Workers:            {NUM_WORKERS} x ({WORKER_CPUS} CPUs, {WORKER_MEMORY_GB} GB)")
print(f"Schedulable CPUs:   {SCHEDULABLE_CPUS} per worker")
print(f"Actors per worker:  {ACTORS_PER_WORKER}")
print(f"Max actors:         {MAX_ACTORS}")
print(f"Chunk max tokens:   {CHUNK_MAX_TOKENS}")
print(f"Embedding model:    {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")
print(f"Milvus:             {MILVUS_HOST}:{MILVUS_PORT}/{COLLECTION_NAME}")

---
## Step 4 — Verify Prerequisites

Check that the PVC exists and Milvus is reachable.

In [ ]:
def verify_prerequisites():
    """Check PVC and Milvus connectivity."""
    configuration = kclient.Configuration()
    configuration.host = API_URL
    configuration.verify_ssl = True
    configuration.api_key["authorization"] = f"Bearer {TOKEN}"

    v1 = kclient.CoreV1Api(kclient.ApiClient(configuration))

    # Check PVC
    print(f"Checking PVC '{PVC_NAME}' in namespace '{NAMESPACE}'...")
    try:
        pvc = v1.read_namespaced_persistent_volume_claim(name=PVC_NAME, namespace=NAMESPACE)
        print(f"  Status:       {pvc.status.phase}")
        access = pvc.spec.access_modes or []
        print(f"  Access mode:  {access[0] if access else 'N/A'}")
    except ApiException as e:
        if e.status == 404:
            print(f"  ERROR: PVC '{PVC_NAME}' not found.")
        else:
            print(f"  ERROR: {e}")

    # Check Milvus
    print(f"\nChecking Milvus at '{MILVUS_HOST}:{MILVUS_PORT}'...")
    try:
        svc = v1.read_namespaced_service(name="milvus-milvus", namespace="milvus")
        print(f"  Service:      {svc.metadata.name} ({svc.spec.cluster_ip})")
        print(f"  Ports:        {[p.port for p in svc.spec.ports]}")
    except ApiException as e:
        if e.status == 404:
            print("  ERROR: Milvus service not found. Deploy Milvus first (see ../milvus-ocp/).")
        else:
            print(f"  ERROR: {e}")

verify_prerequisites()

---
## Step 5 — Submit the RayJob (Parse → Chunk → Embed → Ingest)

This creates a RayJob that:
1. Reads PDFs from the PVC (read-only)
2. Parses each PDF with Docling's `DocumentConverter`
3. Chunks with `HybridChunker` (structure-aware splitting)
4. Embeds chunks with `sentence-transformers`
5. Inserts directly into Milvus

No intermediate files are written to disk.

In [ ]:
# Volume mount — read-only since we only read input PDFs
shared_mount = V1VolumeMount(PVC_MOUNT_PATH, name="shared-data", read_only=True)
data_volume = V1Volume(
    name="shared-data",
    persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(
        claim_name=PVC_NAME, read_only=True,
    ),
)

cluster_config = ManagedClusterConfig(
    head_cpu_requests=HEAD_CPUS,
    head_cpu_limits=HEAD_CPUS,
    head_memory_requests=HEAD_MEMORY_GB,
    head_memory_limits=HEAD_MEMORY_GB,
    num_workers=NUM_WORKERS,
    worker_cpu_requests=WORKER_CPUS,
    worker_cpu_limits=WORKER_CPUS,
    worker_memory_requests=WORKER_MEMORY_GB,
    worker_memory_limits=WORKER_MEMORY_GB,
    volume_mounts=[shared_mount],
    volumes=[data_volume],
    image=IMAGE,
    envs={
        "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.1",
        "RAY_USE_TLS": "0",
    },
)

print("ManagedClusterConfig created.")

In [ ]:
job = RayJob(
    job_name=RAYJOB_NAME,
    entrypoint="python docling_milvus_process.py",
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    runtime_env=RuntimeEnv(
        working_dir="scripts",
        pip=[
            "opencv-python-headless",
            "pypdfium2",
            "pymilvus>=2.4.0",
            "sentence-transformers>=2.2.0",
        ],
        env_vars={
            "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
            "INPUT_PATH": INPUT_PATH,
            "NUM_FILES": str(NUM_FILES),
            "MIN_ACTORS": str(MIN_ACTORS),
            "MAX_ACTORS": str(MAX_ACTORS),
            "CPUS_PER_ACTOR": str(CPUS_PER_ACTOR),
            "BATCH_SIZE": str(BATCH_SIZE),
            "REPARTITION_FACTOR": "10",
            "FILE_TIMEOUT": str(TIMEOUT_SECONDS),
            "MAX_ERRORED_BLOCKS": "50",
            # Milvus
            "MILVUS_HOST": MILVUS_HOST,
            "MILVUS_PORT": str(MILVUS_PORT),
            "MILVUS_DB": MILVUS_DB,
            "MILVUS_COLLECTION": COLLECTION_NAME,
            # Embedding
            "EMBEDDING_MODEL": EMBEDDING_MODEL,
            "EMBEDDING_DIM": str(EMBEDDING_DIM),
            "CHUNK_MAX_TOKENS": str(CHUNK_MAX_TOKENS),
            # Cache dirs
            "HF_HOME": f"{PVC_MOUNT_PATH}/.cache/huggingface",
            "TOKENIZERS_PARALLELISM": "false",
        },
    ),
    ttl_seconds_after_finished=TTL_AFTER_FINISHED,
)

print(f"RayJob '{RAYJOB_NAME}' configured.")

In [ ]:
# Submit the RayJob
job.submit()
print(f"RayJob '{RAYJOB_NAME}' submitted.")

# Patch rayStartParams: head num-cpus=0, workers num-cpus=schedulable
patch = [
    {
        "op": "add",
        "path": "/spec/rayClusterSpec/headGroupSpec/rayStartParams/num-cpus",
        "value": "0",
    },
    {
        "op": "add",
        "path": "/spec/rayClusterSpec/workerGroupSpecs/0/rayStartParams/num-cpus",
        "value": str(SCHEDULABLE_CPUS),
    },
]

print(f"Patching: head num-cpus=0, workers num-cpus={SCHEDULABLE_CPUS}...")
subprocess.run(
    [
        "oc", "patch", "rayjob", RAYJOB_NAME,
        "-n", NAMESPACE,
        "--type", "json",
        "-p", json.dumps(patch),
    ],
    check=True,
)
print("Patch applied.")

---
## Step 6 — Monitor the Job

In [ ]:
job.status()

In [ ]:
!oc get pods -n {NAMESPACE} -l ray.io/cluster={RAYJOB_NAME} --sort-by=.metadata.creationTimestamp

In [ ]:
# Fetch job logs (run after job completes for the processing report)
!oc logs -n {NAMESPACE} -l ray.io/cluster={RAYJOB_NAME},ray.io/node-type=head --tail=100

---
## Step 7 — Verify Milvus Collection

After the RayJob completes, verify that chunks were ingested into Milvus.

In [ ]:
from pymilvus import MilvusClient

# Connect via port-forward or internal service
# For in-cluster notebooks, use the service directly:
milvus_uri = f"http://{MILVUS_HOST}:{MILVUS_PORT}"
# For local notebooks, port-forward first:
#   oc port-forward svc/milvus-milvus 19530:19530 -n milvus
#   milvus_uri = "http://localhost:19530"

milvus_client = MilvusClient(uri=milvus_uri, db_name=MILVUS_DB)

# Check collection exists and get stats
if milvus_client.has_collection(COLLECTION_NAME):
    stats = milvus_client.get_collection_stats(COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' exists.")
    print(f"Stats: {stats}")

    # Sample a few chunks
    milvus_client.load_collection(COLLECTION_NAME)
    results = milvus_client.query(
        collection_name=COLLECTION_NAME,
        filter="chunk_index >= 0",
        output_fields=["source_file", "chunk_index", "text"],
        limit=3,
    )
    print(f"\nSample chunks ({len(results)}):")
    for r in results:
        print(f"  [{r['source_file']}, chunk {r['chunk_index']}]")
        print(f"  {r['text'][:150]}...")
        print()
else:
    print(f"Collection '{COLLECTION_NAME}' not found. Run the RayJob first.")

---
## Step 8 — Deploy a Model on OpenShift AI

Deploy an LLM using vLLM ServingRuntime. This creates a KServe `InferenceService` that exposes an OpenAI-compatible API.

> **Note:** Adjust `MODEL_NAME` and `GPU_COUNT` based on your available resources.
> For CPU-only clusters, use a smaller model like `TinyLlama/TinyLlama-1.1B-Chat-v1.0` with `GPU_COUNT = 0`.

In [ ]:
# If you already have a model deployed, set INFERENCE_URL directly and skip this cell.
# INFERENCE_URL = "https://your-model-route/v1"

isvc_name = MODEL_NAME.split("/")[-1].lower().replace(".", "-")

isvc_manifest = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": isvc_name,
        "namespace": NAMESPACE,
        "annotations": {
            "serving.kserve.io/deploymentMode": "RawDeployment",
        },
    },
    "spec": {
        "predictor": {
            "minReplicas": 1,
            "maxReplicas": 1,
            "model": {
                "modelFormat": {"name": "vllm"},
                "runtime": "vllm-runtime",
                "storageUri": f"hf://{MODEL_NAME}",
                "resources": {
                    "requests": {"cpu": "2", "memory": "8Gi"},
                    "limits": {"cpu": "4", "memory": "16Gi"},
                },
            },
        },
    },
}

if GPU_COUNT > 0:
    isvc_manifest["spec"]["predictor"]["model"]["resources"]["requests"]["nvidia.com/gpu"] = str(GPU_COUNT)
    isvc_manifest["spec"]["predictor"]["model"]["resources"]["limits"]["nvidia.com/gpu"] = str(GPU_COUNT)

# Apply the InferenceService
import tempfile
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(isvc_manifest, f)
    tmp_path = f.name

!oc apply -f {tmp_path} -n {NAMESPACE}
os.unlink(tmp_path)

print(f"\nInferenceService '{isvc_name}' created. Waiting for it to become ready...")
print(f"Monitor with: oc get inferenceservice {isvc_name} -n {NAMESPACE} -w")

In [ ]:
# Get the inference URL once the model is ready
result = subprocess.run(
    ["oc", "get", "inferenceservice", isvc_name, "-n", NAMESPACE,
     "-o", "jsonpath={.status.url}"],
    capture_output=True, text=True,
)
INFERENCE_URL = result.stdout.strip().rstrip("/") + "/v1"
print(f"Inference URL: {INFERENCE_URL}")

---
## Step 9 — RAG Query

Search Milvus for relevant document chunks, build a prompt with context, and query the deployed LLM.

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer

TOP_K = 5

# Load embedding model (same one used during ingestion)
embed_model = SentenceTransformer(EMBEDDING_MODEL)

# Connect to Milvus
milvus_client = MilvusClient(uri=milvus_uri, db_name=MILVUS_DB)
milvus_client.load_collection(COLLECTION_NAME)

# OpenAI-compatible client pointing to vLLM
llm_client = OpenAI(base_url=INFERENCE_URL, api_key="unused")


def rag_query(question: str, top_k: int = TOP_K) -> dict:
    """End-to-end RAG: search Milvus → build prompt → query LLM."""
    # 1. Embed the question
    query_embedding = embed_model.encode([question], normalize_embeddings=True).tolist()

    # 2. Search Milvus
    search_results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=query_embedding,
        limit=top_k,
        output_fields=["source_file", "chunk_index", "text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )

    contexts = []
    for hits in search_results:
        for hit in hits:
            contexts.append({
                "text": hit["entity"]["text"],
                "source_file": hit["entity"]["source_file"],
                "chunk_index": hit["entity"]["chunk_index"],
                "score": hit["distance"],
            })

    # 3. Build RAG prompt
    context_block = "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"score: {c['score']:.3f}]\n{c['text']}"
        for c in contexts
    )

    prompt = (
        "You are a helpful assistant. Answer the user's question based on the "
        "provided context documents. If the answer is not in the context, say so.\n\n"
        f"## Context\n\n{context_block}\n\n"
        f"## Question\n\n{question}\n\n"
        "## Answer\n\n"
    )

    # 4. Query the LLM
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
        temperature=0.1,
    )

    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "contexts": contexts,
    }


print("RAG query function ready.")

In [ ]:
# Try a RAG query
result = rag_query("What are the main topics covered in the documents?")

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['contexts'])}):")
for c in result["contexts"]:
    print(f"  - {c['source_file']} (chunk {c['chunk_index']}, score {c['score']:.3f})")

---
## Step 10 — Cleanup

In [ ]:
# Delete the RayJob (also deletes the associated RayCluster)
# job.delete()

# Delete the InferenceService
# !oc delete inferenceservice {isvc_name} -n {NAMESPACE}

# Drop the Milvus collection
# milvus_client.drop_collection(COLLECTION_NAME)